In [1]:
import nibabel as nib
import numpy as numpy
from scipy.ndimage import median_filter
from skimage import exposure
from skimage.restoration import denoise_bilateral
from scipy.ndimage import zoom
import vtk
from vtk.util import numpy_support
from vtk.util.numpy_support import vtk_to_numpy
import meshio
import sys 
import tomllib
import nrrd
from scipy.ndimage import map_coordinates

from dipy.align.imwarp import SymmetricDiffeomorphicRegistration
from dipy.align.metrics import CCMetric, SSDMetric, EMMetric

sys.path.insert(0, "../src")  # add Folder_2 path to search list
from image_processing_functions import load_vti, load_nifti




image_base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/image/"

input_file_a = image_base + "Left_601 PDT1 0.5 Fat_cropped_denoised.vti"
input_file_b = image_base + "Right_601 PDT1 0.5 Fat_cropped_denoised.vti"

input_file_mask = image_base + "mask.vti"



/opt/homebrew/Caskroom/miniconda/base/envs/eye-project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load segmentation mask (patch)

In [2]:
sequence = "T1"

with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/config.toml", mode="rb") as f:
    config = tomllib.load(f)

base            = config["data"]["base"]
output_folder   = config["data"]["output_folder"]
segment_filename = config["data"]["segment_filename_patch"]
segment_folder  = config["data"]["segment_folder"]

print(base + output_folder + "mesh_offset_"+sequence+".txt")

mesh_offset = numpy.loadtxt(base + output_folder + "mesh_offset_"+sequence+".txt")


/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/mesh/mesh_offset_T1.txt


In [3]:
mask_patch, header_patch = nrrd.read(base + segment_folder + segment_filename)
         # mm per voxel, in array axis order

## Load vti image

In [4]:
data_a, pixel_spacing, origin = load_vti(input_file_a)
data_b, pixel_spacing, origin = load_vti(input_file_b)
mask, spacing_mask, origin_mask = load_vti(input_file_mask)


## Remove patch

In [5]:
rng = numpy.random.default_rng(seed=0)          # fixed seed -> same result every run

patch = mask_patch == 1
n = patch.sum()

data_a = data_a.astype(numpy.float32)           # avoid overflow/clipping in integer types
data_a[patch] = 120 + rng.uniform(-50, 50, size=n)

## Register

In [6]:
# # metric = EMMetric(3, smooth=1.0, q_levels=256)
# # metric = SSDMetric(3, smooth=4)

# metric = CCMetric(3, sigma_diff=2.0, radius=4)

# # metric = CCMetric(3, sigma_diff=4.0, radius=2)   # default sigma_diff=2.0

# sdr = SymmetricDiffeomorphicRegistration(
#     metric,
#     level_iters=[100, 50, 20],   # iterations per resolution level, coarse -> fine
#     step_length=0.25,            # max step per iteration (in voxels)
# )

metric = CCMetric(3, sigma_diff=2.0, radius=8)
sdr = SymmetricDiffeomorphicRegistration(metric, level_iters=[100, 50, 25])

mapping = sdr.optimize(data_a, data_b)




[2026-10-27 15:11:24][dipy] INFO: Creating scale space from the moving image. Levels: 3. Sigma factor: 0.200000.
[2026-10-27 15:11:24][dipy] INFO: Creating scale space from the static image. Levels: 3. Sigma factor: 0.200000.
[2026-10-27 15:11:24][dipy] INFO: Optimizing level 2
[2026-10-27 15:11:29][dipy] INFO: Optimizing level 1
[2026-10-27 15:11:51][dipy] INFO: Optimizing level 0


## Warp

In [7]:
warped_b = mapping.transform(data_b)      # moving b into fixed space
warped_a = mapping.transform_inverse(data_a)
warped_mask = mapping.transform_inverse(mask)

disp_fwd = mapping.get_forward_field()  # shape (X, Y, Z, 3), in voxels
disp_bwd = mapping.get_backward_field()

## Get origin for .vti

In [8]:
segment_origin = numpy.array((181,  98,  35))
new_origin = numpy.array((170, 50, 50))
diff = - new_origin + segment_origin

vti_origin = -mesh_offset - diff*pixel_spacing

In [9]:
print(mesh_offset)
print(pixel_spacing)
print(vti_origin)

[21.49507413 16.82430321 26.72341193]
[0.3125     0.31249997 0.50000006]
[-24.93257413 -31.82430178 -19.22341103]


## Write vti

In [10]:
def write_vti(data, pixel_spacing, offset, output_file):
    data = numpy.asarray(data)

    # Older readers can choke on bool or 64-bit integer arrays
    if data.dtype == bool:
        data = data.astype(numpy.uint8)
    elif data.dtype == numpy.int64:
        data = data.astype(numpy.int32)

    vtk_image = vtk.vtkImageData()
    vtk_image.SetDimensions(data.shape)
    vtk_image.SetSpacing(pixel_spacing)
    vtk_image.SetOrigin(offset[0], offset[1], offset[2])

    # Flatten in Fortran order to match VTK's x-fastest convention
    flat_data = numpy.ascontiguousarray(data.flatten(order="F"))
    vtk_array = numpy_support.numpy_to_vtk(flat_data, deep=True)
    vtk_array.SetName("intensity")
    vtk_image.GetPointData().SetScalars(vtk_array)

    writer = vtk.vtkXMLImageDataWriter()
    writer.SetFileName(output_file)       # should end in .vti
    writer.SetInputData(vtk_image)
    writer.SetDataModeToBinary()          # inline base64 instead of appended raw data
    writer.SetCompressorTypeToNone()      # no zlib compression
    writer.SetHeaderTypeToUInt32()        # 32-bit headers for older readers
    writer.Write()
    print(f"Saved VTI file to: {output_file}")

for data in [warped_b, warped_b]:
    if data.dtype == bool:
        data = data.astype(numpy.uint8)
    elif data.dtype == numpy.int64:
        data = data.astype(numpy.int32)
    elif data.dtype == numpy.float64:
        data = data.astype(numpy.float32)      # GPU-friendly


write_vti(warped_b, pixel_spacing, vti_origin, input_file_b[:-4] + "_warped.vti")
write_vti(warped_a, pixel_spacing, vti_origin, input_file_a[:-4] + "_warped.vti")
write_vti(warped_mask, pixel_spacing, vti_origin, input_file_mask[:-4] + "_warped.vti")

Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/image/Right_601 PDT1 0.5 Fat_cropped_denoised_warped.vti
Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/image/Left_601 PDT1 0.5 Fat_cropped_denoised_warped.vti
Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/image/mask_warped.vti


In [11]:
def warp_points(pts, mapping, spacing, origin):
    vox = (pts - origin) / spacing                      # mm -> voxel in image A
    field = mapping.get_forward_field()                      # (X, Y, Z, 3), voxel units
    disp = numpy.stack([map_coordinates(field[..., k], vox.T, order=1, mode="nearest")
                        for k in range(3)], axis=1)
    return (vox + disp) * spacing + origin               # voxel in B -> mm


mesh_base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/mesh/re-meshed_h_0-9_cenetered_T1/"
mesh_base_out = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/mesh_vti/"
names = ["id_1__h_0-9.stl", "id_2_shell__h_0-9.stl", "id_3__h_0-9.stl", "id_4__h_0-9.stl", "id_7__h_0-9.stl"]


for name in names:
    mesh = meshio.read(mesh_base + name)

    new_points = warp_points(mesh.points.astype(float), mapping,
                            pixel_spacing, origin)

    warped_mesh = meshio.Mesh(points=new_points, cells=mesh.cells)
    meshio.write(mesh_base_out + name, warped_mesh)


/opt/homebrew/Caskroom/miniconda/base/envs/eye-project/lib/python3.13/site-packages/meshio/stl/_stl.py:40: RuntimeWarning: overflow encountered in scalar multiply
  if 84 + num_triangles * 50 == filesize_bytes:


In [12]:
base            = config["data"]["base"]


ref = nib.load(input_file_b[:-4] + ".nii")
ref_affine = ref.affine.copy()

mask_out = numpy.rint( numpy.flip(warped_mask, axis=0)).astype(numpy.uint8)



DEFAULT_COLORS = [
    (0.90, 0.30, 0.30), (0.30, 0.70, 0.30), (0.30, 0.45, 0.90), (0.95, 0.80, 0.20),
    (0.70, 0.35, 0.80), (0.20, 0.80, 0.80), (0.95, 0.55, 0.20), (0.60, 0.60, 0.60),
]

def write_seg_nrrd_from_template(labelmap, affine_ras, example_file, output_file, names=None):
    """
    names: optional dict {label_value: segment_name}; unnamed labels get 'Segment_<label>'.
    """
    labelmap = numpy.rint(labelmap).astype(numpy.uint8)
    _, ex = nrrd.read(example_file)
    names = names or {}

    # Geometry from the NIfTI affine (RAS -> LPS)
    lps = numpy.diag([-1, -1, 1, 1]) @ affine_ras
    header = {
        "space": "left-posterior-superior",
        "space directions": lps[:3, :3].T,
        "space origin": lps[:3, 3],
        "kinds": ["domain", "domain", "domain"],
        "encoding": "gzip",
    }

    # Global segmentation fields from the example (without its geometry)
    for key, value in ex.items():
        if key.startswith("Segmentation_") and key != "Segmentation_ConversionParameters":
            header[key] = value
    header["Segmentation_ContainedRepresentationNames"] = "Binary labelmap|"
    header["Segmentation_ReferenceImageExtentOffset"] = "0 0 0"

    # Per-segment template = the example's Segment0 fields
    template = {k[len("Segment0_"):]: v for k, v in ex.items() if k.startswith("Segment0_")}

    labels = [int(v) for v in numpy.unique(labelmap) if v != 0]
    for i, label in enumerate(labels):
        idx = numpy.argwhere(labelmap == label)
        lo, hi = idx.min(axis=0), idx.max(axis=0)
        color = DEFAULT_COLORS[i % len(DEFAULT_COLORS)]

        fields = dict(template)
        fields.update({
            "ID": f"Segment_{label}",
            "Name": names.get(label, f"Segment_{label}"),
            "NameAutoGenerated": "0",
            "Color": " ".join(f"{c:.6f}" for c in color),
            "ColorAutoGenerated": "0",
            "LabelValue": str(label),
            "Layer": "0",
            "Extent": f"{lo[0]} {hi[0]} {lo[1]} {hi[1]} {lo[2]} {hi[2]}",
        })
        for key, value in fields.items():
            header[f"Segment{i}_{key}"] = value

    nrrd.write(output_file, labelmap, header)



names = {1: "ON", 2: "lens", 3: "eyeball", 4: "lateral",
         5: "medial", 6: "inferior", 7: "superior"}




write_seg_nrrd_from_template(mask_out, ref_affine, base + segment_folder + segment_filename,
                             base + segment_folder + "_warped_mask.seg.nrrd", names=names)




In [13]:
# def pad_to_reference(mask, ref_shape, start):
#     """Place `mask` into a zero array of `ref_shape`, with mask voxel (0,0,0) at index `start`."""
#     ref_shape = numpy.array(ref_shape[:3])
#     start = numpy.array(start, dtype=int)
#     out = numpy.zeros(tuple(ref_shape), dtype=mask.dtype)

#     dst_start = numpy.maximum(start, 0)
#     dst_end = numpy.minimum(start + numpy.array(mask.shape[:3]), ref_shape)
#     if numpy.any(dst_end <= dst_start):
#         raise ValueError("Mask lies completely outside the reference image")
#     src_start, src_end = dst_start - start, dst_end - start

#     out[dst_start[0]:dst_end[0], dst_start[1]:dst_end[1], dst_start[2]:dst_end[2]] = \
#         mask[src_start[0]:src_end[0], src_start[1]:src_end[1], src_start[2]:src_end[2]]
#     if numpy.any(src_start > 0) or numpy.any(src_end < mask.shape[:3]):
#         print("Warning: part of the mask fell outside the reference image and was cut off")
#     return out

# print("mask:", mask_out.shape, "reference:", ref.shape)

# start = (170, 50, 50)

# mask_full = pad_to_reference(mask_out, ref.shape, start)



In [14]:




# idx = numpy.argwhere(mask_out > 0)
# print(mask_out.shape)
# print(idx)

# if idx.size:
#     print("labelled region in mask_out:", idx.min(axis=0), "to", idx.max(axis=0))